In [0]:
import json
from pyspark.sql import Row
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Fact Pitching

In [0]:
def convert_innings_pitched(ip_col):
    """
    MLB's innings_pitched format: the digit after the decimal is OUTS, not tenths. .0 = 0 outs, .1 = 1 out (1/3 inning), .2 = 2 outs (2/3). EX: 6.1 = 6 full innings + 1 out
    """
    whole = split(ip_col, "\\.").getItem(0).cast("double")
    fraction_digit = coalesce(split(ip_col, "\\.").getItem(1), lit("0")).cast("double")
    outs_as_innings = fraction_digit / 3.0
    
    return whole + outs_as_innings

In [0]:
pitching = spark.table("silver.mlb_pitching_stats")

fact_pitching = (
    pitching
    .withColumn("stat_type", lit("pitching"))
    .withColumn("innings_pitched_raw", col("innings_pitched"))
    .withColumn("innings_pitched_decimal", convert_innings_pitched(col("innings_pitched")))
)

In [0]:
games_lookup = spark.table("silver.mlb_schedule").select("game_pk", "game_date", "home_team_id", "away_team_id")

pitching_enriched = (
    fact_pitching
    .join(games_lookup, on="game_pk", how="left")
    .withColumn(
        "oppenent_team_id",
        when(col("team_id") == col("home_team_id"), col("away_team_id"))
        .otherwise(col("home_team_id"))
    )
)



# Sanity Check: any fact rows that failed to find a matching game

In [0]:
missing_game_pitching = pitching_enriched.filter(col("game_date").isNull()).count()

print(f"pitching rows with no matching game: {missing_game_pitching}")

# Writing into Gold Layer

In [0]:
pitching_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("game_date") \
    .saveAsTable("gold.fact_pitching_stats")